In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms


# --- Get images ---

# Note: Standard image size is 640 x 640

# Tensor setup
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# Load dataset
full_dataset = datasets.ImageFolder(
    root="Dataset/",
    transform=transform
)

print("Detected classes:", full_dataset.class_to_idx)

# 80/20 Train / Test split
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

dataloader_train = DataLoader(train_dataset, batch_size=32, shuffle=True)
dataloader_test = DataLoader(test_dataset, batch_size=32, shuffle=False)

Detected classes: {'Crack': 0, 'Pothole': 1, 'Surface Erosion': 2}


### Make CNN

In [44]:
from torch import nn
import torch.nn.functional as F


# --- Make CNN ---

class CNN(nn.Module):
    def __init__(self, in_channels=3, num_classes=3):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=8, kernel_size=7, padding=3)

        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=5, padding=2)

        self.conv3 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((7, 7))
        self.fc1 = nn.Linear(32 * 7 * 7, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.adaptive_pool(x)
        x = x.reshape(x.shape[0], -1)
        x = self.fc1(x)
        return x




### Training

In [45]:
from torch import optim

# --- Train CNN ---

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = CNN(in_channels=3, num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs=10
total_batches = len(dataloader_train)

for epoch in range(num_epochs):
  # Iterate over training batches

  for batch_index, (data, targets) in enumerate(dataloader_train):
    data = data.to(device)
    targets = targets.to(device)
    scores = model(data)
    loss = criterion(scores, targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (batch_index + 1) % 5 == 0 or (batch_index + 1) == total_batches:
      print(f"Epoch [{epoch + 1}/{num_epochs}]")
      print(f"Batch [{batch_index + 1}/{total_batches}]")
      print(f"Loss: {loss.item():.4f}")


Device: cpu
Epoch [1/10]
Batch [5/39]
Loss: 1.8110
Epoch [1/10]
Batch [10/39]
Loss: 1.0333
Epoch [1/10]
Batch [15/39]
Loss: 1.0581
Epoch [1/10]
Batch [20/39]
Loss: 1.2211
Epoch [1/10]
Batch [25/39]
Loss: 0.9769
Epoch [1/10]
Batch [30/39]
Loss: 1.0691
Epoch [1/10]
Batch [35/39]
Loss: 0.9963
Epoch [1/10]
Batch [39/39]
Loss: 0.9700
Epoch [2/10]
Batch [5/39]
Loss: 1.2152
Epoch [2/10]
Batch [10/39]
Loss: 0.9147
Epoch [2/10]
Batch [15/39]
Loss: 1.0362
Epoch [2/10]
Batch [20/39]
Loss: 0.9396
Epoch [2/10]
Batch [25/39]
Loss: 0.9619
Epoch [2/10]
Batch [30/39]
Loss: 0.9872
Epoch [2/10]
Batch [35/39]
Loss: 0.9355
Epoch [2/10]
Batch [39/39]
Loss: 0.9193
Epoch [3/10]
Batch [5/39]
Loss: 0.8810
Epoch [3/10]
Batch [10/39]
Loss: 1.0454
Epoch [3/10]
Batch [15/39]
Loss: 0.8601
Epoch [3/10]
Batch [20/39]
Loss: 0.9514
Epoch [3/10]
Batch [25/39]
Loss: 0.6721
Epoch [3/10]
Batch [30/39]
Loss: 0.8288
Epoch [3/10]
Batch [35/39]
Loss: 0.8925
Epoch [3/10]
Batch [39/39]
Loss: 0.6099
Epoch [4/10]
Batch [5/39]
Loss:

### Evaluation

In [47]:
from torchmetrics import Accuracy, Precision, Recall

# --- Evaluate CNN ---
acc = Accuracy(task="multiclass",num_classes=10)
precision = Precision(task="multiclass", num_classes=10, average="macro")
recall = Recall(task="multiclass", num_classes=10, average="macro")

model.eval()
with torch.no_grad():
    for images, labels in dataloader_test:
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        # Update
        acc(preds, labels)
        precision(preds, labels)
        recall(preds, labels)

print(f"Test Accuracy:  {acc.compute():.4f}")
print(f"Test Precision: {precision.compute():.4f}")
print(f"Test Recall:    {recall.compute():.4f}")

Test Accuracy:  0.7810
Test Precision: 0.7783
Test Recall:    0.7278
